In [0]:
# %pip install -q pandas transformers torch tqdm

Lyrics Emotion Classification Pipeline
=======================================
Input:  analysis_df with columns: rank, artist, title, region,
        spotify_uri, lyrics_in_en, original_lang
Output: analysis_df + 13 emotion score columns + dominant_emotion column

In [0]:
import re
import pandas as pd
from tqdm import tqdm
from transformers import pipeline
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent  # go up from notebooks to project root

# Load from 03_lyrics_trans (emotional analysis ready - all lyrics translated to English)
# analysis_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "03_lyrics_trans.csv")
# DATABRICKS PATH
analysis_df = pd.read_csv(Path("/Volumes/songs_db/default/storage/03_lyrics_trans.csv"))

if "lyrics_in_en" not in analysis_df.columns:
    raise KeyError("Expected 'lyrics_in_en' column in 03_lyrics_trans.csv")

print(f'Loaded {len(analysis_df)} songs')
print(f'Columns: {list(analysis_df.columns)}')

# display(analysis_df.head(3))

In [0]:
# Config

## TODO: angst, party, other, solitude, relaxing, aren't very good indicators.
# re look at how the model classify the lyrics / chunks. 
# compaction of context into an MD file, then redo whole analysis with more regions.

EMOTIONS = [
    "love", "longing", "joy", "heartbreak",
    "despair", "hope", "lonely",
    "sensual", "grief", "anger"
]
 
# How many words per chunk (stay under ~400 tokens)
CHUNK_WORDS = 350
 
# Score threshold: below this, treat as 0 (noise floor for multi-label)
NOISE_FLOOR = 0.10

# Checkpointing
checkpoint_every = 10
# checkpoint_path = PROJECT_ROOT / "data" / "processed" / "emotion_scores_checkpoint.csv"
# DATABRICKS PATH
checkpoint_path = Path("/Volumes/songs_db/default/storage/emotion_scores_checkpoint.csv")

### Clean lyrics

In [0]:
# ─────────────────────────────────────────────
# 1. CLEANING  — keep punctuation & case
# ─────────────────────────────────────────────
 
def clean_lyrics(text: str, artist: str = '') -> str:
    """
    Remove structural noise but preserve punctuation and casing.
    Punctuation (! ? ...) and capitalisation carry emotional signal
    for NLI-based classifiers — don't strip them.
 
    artist: the artist string from the dataframe row. When provided,
    the first line is dropped only if its tokens are a subset of the
    known artist names — much more precise than a regex heuristic.
    Falls back to the regex heuristic when artist is unavailable.
    """
    if not isinstance(text, str) or not text.strip():
        return ''
 
    def is_artist_credit(line: str) -> bool:
        """
        True if every name token in `line` exists in the artist pool.
        Both strings are split on commas, ampersands, and feat/ft.
 
            artist = "Jason, Bonnie"          line = "Jason"          → True
            artist = "ARIA VEGA, Ryan Castro" line = "Ryan Castro"    → True
            artist = "Jason, Bonnie"          line = "Baby come back" → False
        """
        splitter = r'[,&]|\bfeat\.?\b|\bft\.?\b'
        artist_tokens = {
            t.strip().lower()
            for t in re.split(splitter, artist, flags=re.IGNORECASE)
            if t.strip()
        }
        line_tokens = {
            t.strip().lower()
            for t in re.split(splitter, line, flags=re.IGNORECASE)
            if t.strip()
        }
        return bool(line_tokens) and line_tokens.issubset(artist_tokens)
 
    # Remove section headers: [Verse 1], [Chorus], [Bridge] etc.
    text = re.sub(r'\[[^\]]*\]', '', text)
 
    # Remove repetition annotations: (x3), (×2), (2x)
    text = re.sub(r'\([\d×xX]+\)', '', text)
 
    # Remove leading artist/feature credits that sometimes appear
    # at the top of translated lyrics (e.g. "ARIA VEGA, Ryan Castro\n").
    lines = text.strip().splitlines()
    if lines:
        first = lines[0].strip()
        if artist and is_artist_credit(first):
            lines = lines[1:]
        elif not artist and re.match(r'^[A-Za-z\s,&]+$', first) and len(first) < 80:
            lines = lines[1:]
    text = '\n'.join(lines)
 
    # Collapse excessive blank lines but keep single line breaks
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
 
    return text.strip()
 
 
def dedupe_lines(text: str) -> str:
    """
    Remove duplicate lines (repeated choruses inflate scores).
    Keeps first occurrence; preserves order.
    """
    seen = set()
    result = []
    for line in text.splitlines():
        key = line.strip().lower()
        if key and key not in seen:
            seen.add(key)
            result.append(line.strip())
    return ' '.join(result)
 
 
def prepare_lyrics(text: str, artist: str = '') -> str:
    return dedupe_lines(clean_lyrics(text, artist=artist))

### Chunking

In [0]:
# ─────────────────────────────────────────────
# 2. CHUNKING  — handle long lyrics gracefully
# ─────────────────────────────────────────────
 
def chunk_text(text: str, chunk_words: int = CHUNK_WORDS) -> list[str]:
    """Split text into word-count chunks with a small overlap."""
    words = text.split()
    if not words:
        return []
    overlap = 30
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_words
        chunks.append(' '.join(words[start:end]))
        start = end - overlap  # small overlap so context isn't lost at boundaries
        if start >= len(words):
            break
    return chunks

### Classification

In [0]:
# ─────────────────────────────────────────────
# 3. CLASSIFICATION
# ─────────────────────────────────────────────
 
def load_classifier():
    """
    facebook/bart-large-mnli  — best general zero-shot NLI classifier.
    Falls back to a smaller model if memory is tight.
    Switch to cross-encoder/nli-deberta-v3-large for higher accuracy
    at the cost of ~2× slower inference.
    """
    print("Loading zero-shot classifier (bart-large-mnli)...")
    return pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=0  # set to -1 to force CPU; 0 for first GPU
    )
 
 
def classify_song(lyrics: str, classifier) -> dict:
    """
    Classify a single song's lyrics.
    Chunks are batched together in one classifier call for speed.
    Returns dict {emotion: score}.
    """
    BATCH_SIZE = 16
 
    if not isinstance(lyrics, str) or not lyrics.strip():
        return {e: 0.0 for e in EMOTIONS}
 
    chunks = chunk_text(lyrics)
    raw_results = []
 
    for i in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[i : i + BATCH_SIZE]
        raw_results.extend(
            classifier(batch, candidate_labels=EMOTIONS, multi_label=True)
        )
 
    all_scores = [dict(zip(r["labels"], r["scores"])) for r in raw_results]
 
    avg = {
        e: round(sum(s[e] for s in all_scores) / len(all_scores), 4)
        for e in EMOTIONS
    }
    return {e: (v if v >= NOISE_FLOOR else 0.0) for e, v in avg.items()}
 
 
def load_checkpoint(checkpoint_path: Path) -> pd.DataFrame | None:
    """
    Load an existing checkpoint CSV if it exists, else return None.
    The checkpoint contains all original columns + emotion score columns
    for every song that has already been classified.
    """
    if checkpoint_path.exists():
        df = pd.read_csv(checkpoint_path)
        emotion_cols = [f"emotion_{e}" for e in EMOTIONS]
        already_done = df[emotion_cols].notna().all(axis=1).sum()
        print(f"Checkpoint found: {already_done} / {len(df)} songs already classified.")
        return df
    return None
 
 
def save_checkpoint(df: pd.DataFrame, checkpoint_path: Path) -> None:
    """Write the current state of df (including any NaN emotion cols) to disk."""
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(checkpoint_path, index=False)

### Main Pipeline

In [0]:
# ── Step 2.5: prepare dataframe state for classification ─────────────────────
emotion_cols = [f"emotion_{e}" for e in EMOTIONS]
df = load_checkpoint(checkpoint_path)

if df is None:
    df = analysis_df.copy()
    for col in emotion_cols:
        if col not in df.columns:
            df[col] = pd.NA

if 'lyrics_prepared' not in df.columns:
    df['lyrics_prepared'] = df.apply(
        lambda r: prepare_lyrics(r.get('lyrics_in_en', ''), artist=r.get('artist', '')),
        axis=1
    )

needs_classification = df[emotion_cols].isna().any(axis=1)
todo = df[needs_classification].index.tolist()
print(f"Songs pending classification: {len(todo)} / {len(df)}")

# ── Step 3: classify only missing rows + checkpointing ─────────────────────
if len(todo) == 0:
    print("No unclassified rows. Using existing scores.")
else:
    try:
        classifier = load_classifier()
    except Exception as e:
        print(f"Primary model load failed ({e}); retrying on CPU...")
        classifier = pipeline(
            "zero-shot-classification",
            model="facebook/bart-large-mnli",
            device=-1
        )
    n_saved = 0

    for i, idx in enumerate(tqdm(todo, desc="Classifying songs", unit="song")):
        lyrics = df.at[idx, 'lyrics_prepared']
        scores = classify_song(lyrics, classifier)

        for emotion, score in scores.items():
            df.at[idx, f"emotion_{emotion}"] = score

        # Flush to disk every checkpoint_every songs
        n_saved += 1
        if n_saved % checkpoint_every == 0:
            save_checkpoint(df, checkpoint_path)
            tqdm.write(f"  ✓ Checkpoint saved ({n_saved} songs classified this run)")

    # Final save after the loop completes
    save_checkpoint(df, checkpoint_path)
    print(f"Done. Checkpoint saved to {checkpoint_path}")

# ── Step 4: derive dominant emotion ──────────────────────────────────
df['dominant_emotion'] = df[emotion_cols].idxmax(axis=1).str.replace('emotion_', '', regex=False)
df['dominant_emotion'] = df['dominant_emotion'].where(
    df[emotion_cols].max(axis=1) > NOISE_FLOOR, other='unclassified'
)

df = df.drop(columns=['lyrics_prepared'])

# Output to emotion_scores.csv with appropriate schema
# final_output_path = PROJECT_ROOT / "data" / "processed" / "emotion_scores.csv"
#DATABRICKS PATH
final_output_path = Path("/Volumes/songs_db/default/storage/emotion_scores.csv")

df.to_csv(final_output_path, index=False)
print(f"Saved emotion analysis to {final_output_path}")
display(df.head(3))

In [0]:
def regional_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Average emotion scores per region.
    Useful for radar/heatmap visualisation.
    """
    emotion_cols = [f"emotion_{e}" for e in EMOTIONS]
    summary = df.groupby('region')[emotion_cols].mean().round(3)
    summary.columns = [c.replace('emotion_', '') for c in summary.columns]
    return summary

summary = regional_summary(df)
display(summary)
# summary.to_csv(PROJECT_ROOT / "data" / "processed" / "regional_summary.csv", index=False)
# DATABRICKS PATH
summary.to_csv("/Volumes/songs_db/default/storage/regional_summary.csv", index=False)
